# **Köşeleri Bulmak**

####**Bu derste şunları öğreneceğiz:**
1. Köşeleri bulmak için Harris Köşelerini kullanmak
2. Shi-Tomasi köşe algılama


In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt

def imshow(title = "Image", image = None, size = 10):
    w, h = image.shape[0], image.shape[1]
    aspect_ratio = w/h
    plt.figure(figsize=(size * aspect_ratio,size))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.show()


## **Köşe Nedir?**

Köşe, yerel komşuluğu iki baskın ve farklı kenar yönünde duran bir noktadır. Başka bir deyişle, bir köşe iki kenarın birleşimi olarak yorumlanabilir; burada bir kenar görüntü parlaklığındaki ani bir değişikliktir. Köşeler görüntüdeki önemli özelliklerdir ve genellikle öteleme, döndürme ve aydınlatmaya karşı değişmez olan ilgi noktaları olarak adlandırılırlar.\
<img src="korner.jpg" width="600">

### **Harris Köşe Tespiti** 1988 yılında köşe tespiti için geliştirilmiş ve oldukça iyi çalışan bir algoritmadır.


**Makale** - http://www.bmva.org/bmvc/1988/avc-88-023.pdf

**cv2.cornerHarris**(input image, block size, ksize, k)
- Input image - gri tonlamalı ve float32 tipinde olmalıdır.
- blockSize - köşe tespiti için dikkate alınan komşuluk boyutu
- ksize - kullanılan Sobel türevinin açıklık parametresidir.
- k - harris dedektörü denklemdeki serbest parametre
- **Output** – köşe konumları dizisi (x,y)




In [ ]:
# Görüntüyü yükle, ardından gri tonlama
image = cv2.imread('../files/images/deneme7.JPG')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# cornerHarris işlevi dizi veri türünün float32 olmasını gerektirir
gray = np.float32(gray)

harris_corners = cv2.cornerHarris(gray, 3, 3, 0.05)

#Köşe noktalarını büyütmek için dilatasyon kullanıyoruz\
kernel = np.ones((7,7),np.uint8)
harris_corners = cv2.dilate(harris_corners, kernel, iterations = 2)

# Optimum değer için eşik, görüntüye bağlı olarak değişebilir.
image[harris_corners > 0.025 * harris_corners.max() ] = [255, 127, 127]
#harris_corners.max() Harris yanıt haritasındaki en büyük değeri bulur. Bu, en güçlü köşe yanıtıdır.
#0.025 * harris_corners.max() Maksimum değerin %2.5’i hesaplanır.
# Bu, bir eşik değeri (threshold) olarak kullanılır. 
#Yani köşe yanıtı bu değerin üzerinde olan pikseller “köşe” olarak kabul edilir.
#harris_corners > 0.025 * harris_corners.max()
#Mantıksal bir maske üretir (aynı boyutta True/False dizisi).
#True olanlar, köşe kabul edilen piksellerdir.
#image[...] = [255, 127, 127]
#Bu maskede True olan tüm pikseller, orijinal görüntü üzerinde [255, 127, 127] (BGR) rengi ile boyanır.

#Kodlama nasıl yapılmış
#Boolean Maskesi --> harris_corners > 0.025 * harris_corners.max()
# harris_corners ile aynı boyutta bir True/False dizisi üretir.
#True olan konumlar, köşe olarak algılanan piksellerdir.
# İndeksleme -->image[mask] ifadesi, maskenin True olduğu tüm pikselleri seçer.
#Bu işlem, üç kanallı (BGR) görüntüde her bir pikselin tüm kanallarını aynı anda hedefler.
# Atama  -->= [255, 127, 127]

imshow('Harris Corners', image,5)

## Shi-Tomasi köşe algılama
**cv2.goodFeaturesToTrack**(input image, maxCorners, qualityLevel, minDistance)\
cv2.goodFeaturesToTrack() fonksiyonu, Shi-Tomasi köşe algılama yöntemini kullanarak görüntüdeki güçlü köşeleri bulur.\
Harris’e benzer ama daha güvenilir sonuçlar verir çünkü sadece minimum özdeğer’e (eigenvalue) odaklanır.

corners = cv2.goodFeaturesToTrack(\
    image,            # Gri tonlamalı görüntü (uint8 veya float32)\
    maxCorners,       # Döndürülecek maksimum köşe sayısı \
    qualityLevel,     # Minimum kabul edilebilir kalite (0–1 arası)\
    minDistance       # İki köşe arasındaki minimum mesafe (piksel)\
)


In [ ]:
img = cv2.imread('../files/images/deneme7.JPG')
gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)

# We specific the top 50 corners
corners = cv2.goodFeaturesToTrack(gray, 150, 0.0005, 10)

for corner in corners:
    x, y = corner[0]
    x = int(x)
    y = int(y)
    cv2.rectangle(img,(x-10,y-10),(x+10,y+10),(0,255,0), 2)
    
imshow("Corners Found", img)